In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

RANDOM_STATE = 42
TEST_SIZE = 0.2

In [7]:
PATH = r"C:\ML\data\model_ready_scaled.csv"

df = pd.read_csv(PATH)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (200000, 67)


,diagnosis_year,age_midpoint,grade_unified_num,stage_ordinal,tumor_size_mm,log1p_tumor_size_mm,nodes_positive_count,nodes_examined_count,nodes_positive_ratio,receptor_unknown_or_unavailable_count,...,race_Asian or Pacific Islander,race_Black,race_Unknown,race_White,laterality_bilateral_single_primary,laterality_left,laterality_one_side_unspecified,laterality_paired_site_laterality_unknown,laterality_right,survive_after_5
0,0.561048,-0.711807,-1.550684,-0.594616,-0.214331,0.083366,-0.327935,0.608146,-0.428783,-0.812682,...,0,0,0,1,0,0,0,0,1,1
1,0.561048,0.000082,1.234184,2.875063,-0.214331,0.083366,0.004350,0.899825,-0.070402,-0.812682,...,0,0,0,1,0,1,0,0,0,1
2,0.210869,-0.355862,1.234184,-0.594616,-0.441698,-0.251876,-0.327935,0.024787,-0.428783,-0.812682,...,0,1,0,0,0,1,0,0,0,1
3,-0.314401,1.423859,-0.158250,-0.594616,-0.725907,-0.826898,-0.327935,2.795741,-0.428783,0.363167,...,0,0,0,1,0,0,0,0,1,1
4,-0.839671,0.711970,-0.158250,-0.594616,-0.839591,-1.143356,-0.327935,0.462306,-0.428783,2.714866,...,0,0,0,1,0,1,0,0,0,1


In [8]:
X = df.drop(columns=["survive_after_5"])

y = df["survive_after_5"]

print(X.shape)
print(y.value_counts())

(200000, 66)
survive_after_5
1    156973
0     43027
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

Train shape: (160000, 66)
Test shape : (40000, 66)

Train class distribution:
survive_after_5
1    125578
0     34422
Name: count, dtype: int64

Test class distribution:
survive_after_5
1    31395
0     8605
Name: count, dtype: int64


In [10]:
class LR:
    def __init__(self, lr = 0.01, n_iters = 5000):
        self.lr = lr
        self.n_iters = n_iters  
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            linear_output = np.dot(X, self.weights) + self.bias
            y_pred_proba = self.sigmoid(linear_output)

            dw = (1 / n_samples) * np.dot(X.T, (y_pred_proba - y))
            db = (1 / n_samples) * np.sum(y_pred_proba - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            eps = 1e-15
            loss = -np.mean(
                y * np.log(y_pred_proba + eps) +
                (1 - y) * np.log(1 - y_pred_proba + eps)
            )

            self.loss_history.append(loss)

    def predict_proba(self, X):
        X = np.asarray(X)
        linear_output = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_output)
    
    def predict(self, X, threshold=0.5):
        proba_malignant = self.predict_proba(X)
        return np.where(proba_malignant >= threshold, 0, 1)


class NB:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        self.classes = None
        self.class_priors = {}
        self.means = {}
        self.vars = {}

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        self.classes = np.unique(y)

        for cls in self.classes:
            X_cls = X[y == cls]

            self.class_priors[cls] = X_cls.shape[0] / X.shape[0]
            self.means[cls] = np.mean(X_cls, axis=0)
            self.vars[cls] = np.var(X_cls, axis=0) + self.var_smoothing

    def _log_gaussian_density(self, X, cls):
        mean = self.means[cls]
        var = self.vars[cls]

        log_density = -0.5 * np.log(2 * np.pi * var) - ((X - mean) ** 2) / (2 * var)

        return np.sum(log_density, axis=1)

    def predict_log_proba(self, X):
        X = np.asarray(X)

        log_posteriors = []

        for cls in self.classes:
            log_prior = np.log(self.class_priors[cls])
            log_likelihood = self._log_gaussian_density(X, cls)
            log_posterior = log_prior + log_likelihood
            log_posteriors.append(log_posterior)

        return np.vstack(log_posteriors).T

    def predict(self, X):
        log_posteriors = self.predict_log_proba(X)
        class_indices = np.argmax(log_posteriors, axis=1)
        return self.classes[class_indices]

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
lr = LR(
    lr=0.01,
    n_iters=5000
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

In [13]:
nb = NB()

nb.fit(X_train_scaled, y_train)

y_pred_nb = nb.predict(X_test_scaled)

In [14]:
print("Logistic Regression Classification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["+", "-"],
    digits=4
))

print(confusion_matrix(y_test, y_pred_lr, labels=[0, 1]))

print("Naive Bayes Classification Report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["+", "-"],
    digits=4
))

print(confusion_matrix(y_test, y_pred_nb, labels=[0, 1]))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           +     0.1308    0.5250    0.2094      8605
           -     0.2516    0.0438    0.0746     31395

    accuracy                         0.1473     40000
   macro avg     0.1912    0.2844    0.1420     40000
weighted avg     0.2256    0.1473    0.1036     40000

[[ 4518  4087]
 [30021  1374]]
Naive Bayes Classification Report:
              precision    recall  f1-score   support

           +     0.6021    0.5138    0.5544      8605
           -     0.8719    0.9069    0.8891     31395

    accuracy                         0.8224     40000
   macro avg     0.7370    0.7103    0.7217     40000
weighted avg     0.8138    0.8224    0.8171     40000

[[ 4421  4184]
 [ 2922 28473]]
